# Experiment 8 — Patient/Eager, zones [2.5%, 5%, 10%, 100%], verbose=2

Same P-then-E controlled round as Experiments 6-7, this time with an even
more aggressive starting zone (2.5%, ~10.5K rows) and **verbose=2**, so every
cell prints the full live decision narration as pattern_search_cv makes it -
climber calibration, ring crossings, sweep probes, pattern moves,
contractions, data climbs, merges. Run the cells yourself in Jupyter/VS Code
to watch it stream in real time. subsample='stratified' (explicit, matching
Experiment 7's winning configuration); poll='opportunistic' (explicit, matches
what 'auto' already resolves to on this machine).

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_8820\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 4.4 s


In [2]:
# --- common setup: grid, CV, zones ladder [2.5%, 5%, 10%, 100%] ---
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from pattern_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
ZONES = [0.025, 0.05, 0.1, 1.0]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)

def run_policy(policy):
    search = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, poll="opportunistic",
                             contraction=policy, data_zones=ZONES,
                             subsample="stratified",
                             random_state=0, verbose=2)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    res = search.cv_results_
    evals = []
    for p, sc, nr, ft in zip(res["params"], res["mean_test_score"],
                             res["n_resources"], res["mean_fit_time"]):
        key = (p["max_features"], p["n_estimators"], p["max_depth"], int(nr))
        evals.append({"key": key, "mae": -sc, "fit_work": float(ft) * 5})
    return {
        "policy": policy, "wall": wall,
        "n_fits": len(res["params"]),
        "equiv": float(np.sum(np.asarray(res["n_resources"]) / len(y))),
        "best": search.best_params_, "best_mae": -search.best_score_,
        "zones_used": sorted(set(int(v) for v in res["n_resources"])),
        "fit_work_total": sum(e["fit_work"] for e in evals),
        "evals": evals,
    }


In [3]:
# --- PATIENT run: watch the [pattern_search_cv] lines stream below ---
P = run_policy("patient")
print()
print(f"PATIENT: {P['n_fits']} evals, {P['equiv']:.2f} equiv, "
      f"{P['wall']:.1f} s wall, best {P['best']} MAE {P['best_mae']:.3f}")

[pattern_search_cv] PatternSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [10.51086569 18.87057757 27.78200459 39.97915506 53.15509057]


[pattern_search_cv] fit_time: 30.059539


[pattern_search_cv] score_time per fold: [1.52807283 1.45751905 1.50580978 1.52522922 1.51085591]


[pattern_search_cv] score_time: 1.505497



PATIENT: 22 evals, 5.43 equiv, 626.1 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [4]:
# --- EAGER run ---
E = run_policy("eager")
print()
print(f"EAGER: {E['n_fits']} evals, {E['equiv']:.2f} equiv, "
      f"{E['wall']:.1f} s wall, best {E['best']} MAE {E['best_mae']:.3f}")

[pattern_search_cv] PatternSearchCV: optimizing metric = neg_mean_absolute_error


[pattern_search_cv] cv = TimeSeriesSplit


[pattern_search_cv]   max_features : [2, 3, 4]


[pattern_search_cv]   n_estimators : [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260]


[pattern_search_cv]   max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


...

[pattern_search_cv] fit_time per fold: [11.19089127 21.72018957 33.50850368 40.23781681 49.76190233]


[pattern_search_cv] fit_time: 31.283861


[pattern_search_cv] score_time per fold: [2.59298658 1.42408276 2.61211395 1.44787908 1.51800585]


[pattern_search_cv] score_time: 1.919014



EAGER: 22 evals, 5.43 equiv, 649.9 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [5]:
# --- P vs E comparison, zones [2.5, 5, 10, 100]% ---
print("=" * 76)
print(f"zones ladder {ZONES} (subsample='stratified')")
print(f"{'':22s}{'PATIENT':>14s}{'EAGER':>14s}")
print(f"{'evaluations':22s}{P['n_fits']:>14d}{E['n_fits']:>14d}")
print(f"{'full-fit equivalents':22s}{P['equiv']:>14.2f}{E['equiv']:>14.2f}")
print(f"{'wall-clock (s)':22s}{P['wall']:>14.1f}{E['wall']:>14.1f}")
print(f"{'P/E wall-clock ratio':22s}{'':>14s}{P['wall'] / E['wall']:>14.3f}")
print(f"{'summed fit work (s)':22s}{P['fit_work_total']:>14.1f}{E['fit_work_total']:>14.1f}")
print(f"{'best point':22s}{str(tuple(P['best'].values())):>14s}{str(tuple(E['best'].values())):>14s}")
print(f"{'best CV MAE':22s}{P['best_mae']:>14.3f}{E['best_mae']:>14.3f}")
print(f"{'zones used (rows)':22s}{str(P['zones_used']):>14s}{str(E['zones_used']):>14s}")
print()

# machine-drift control: identical (params, rows) evaluated in BOTH runs
p_map = {e["key"]: e["fit_work"] for e in P["evals"]}
e_map = {e["key"]: e["fit_work"] for e in E["evals"]}
shared = sorted(set(p_map) & set(e_map))
print(f"shared evaluations: {len(shared)} of {P['n_fits']}/{E['n_fits']}")
if shared:
    sum_p = sum(p_map[k] for k in shared)
    sum_e = sum(e_map[k] for k in shared)
    print(f"sum fit-work over shared evals: P={sum_p:.1f}s E={sum_e:.1f}s "
          f"E/P(sum ratio)={sum_e / sum_p:.3f}")
    print("  (this sum-based ratio is the one consistent with wall-clock "
          "direction - do NOT use median-of-per-eval-ratios, see Exp 6/7 "
          "correction in EXPERIMENTS.md)")
print(f"unique-to-patient evals: {sorted(set(p_map) - set(e_map))}")
print(f"unique-to-eager evals  : {sorted(set(e_map) - set(p_map))}")


zones ladder [0.025, 0.05, 0.1, 1.0] (subsample='stratified')
                             PATIENT         EAGER
evaluations                       22            22
full-fit equivalents            5.43          5.43
wall-clock (s)                 626.1         649.9
P/E wall-clock ratio                         0.963
summed fit work (s)           1254.3        1282.8
best point              (4, 130, 17)  (4, 130, 17)
best CV MAE                  805.038       805.038
zones used (rows)     [10461, 418416][10461, 418416]

shared evaluations: 22 of 22/22
sum fit-work over shared evals: P=1254.3s E=1282.8s E/P(sum ratio)=1.023
  (this sum-based ratio is the one consistent with wall-clock direction - do NOT use median-of-per-eval-ratios, see Exp 6/7 correction in EXPERIMENTS.md)
unique-to-patient evals: []
unique-to-eager evals  : []
